# few_shot_check
Each condition vs K=0 (zero_ref), uid-paired. domain vs mismatched = is the disruption specific to domain-relevant exemplars; domain vs filler = real exemplar block vs equal-token neutral filler, so the non-length component is semantics plus exemplar block structure; anti-render vs domain = whether "do not draw" framing helps; K2 vs K5 = dose.
Leakage (H1): validation-gated CLIP — validate image-text CLIP on K=0 first, then measure CLIP(K image, exemplar texts) - CLIP(K0 image, same). PF-drop + eyeball are primary; CLIP is a gated supporting signal.
Verdict: H1 leak / H2 ignore / H3 abstract.

In [ ]:
from google.colab import drive
import os, json, math, statistics as st
from scipy.stats import ttest_rel, t as tdist
drive.mount("/content/drive")
DRIVE  = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SCORES = f"{DRIVE}/scores"
def load(run):
    d={}
    for l in open(f"{SCORES}/test__{run}.jsonl", encoding="utf-8"):
        r=json.loads(l)
        if "error" not in r: d[r["uid"]]=r
    return d
K0 = load("zero_ref")
runs = {r: load(r) for r in ["k2_domain","k5_domain","k5_mismatched","k5_filler","k5_antirender"]}
print("K=0 (zero_ref):", len(K0), "| " + " | ".join(f"{k}:{len(v)}" for k,v in runs.items()))

Mounted at /content/drive
K=0 (zero_ref): 488 | k2_domain:488 | k5_domain:488 | k5_mismatched:488 | k5_filler:488 | k5_antirender:488


## Metrics vs K=0 (H2 ignore / H3 abstract), plus the filler (length) and anti-render controls

In [ ]:
ov = lambda d: math.sqrt(max(d["PF"],0)*max(d["SC"],0))
def paired(cur, ref, metric):
    I=sorted(set(cur)&set(ref)); d=[metric(cur[u])-metric(ref[u]) for u in I]
    n=len(d); mdv=st.mean(d); se=st.stdev(d)/math.sqrt(n); t,p=ttest_rel([metric(cur[u]) for u in I],[metric(ref[u]) for u in I])
    tc=tdist.ppf(0.95,n-1); return mdv,p,mdv-tc*se,mdv+tc*se,n
print("each condition minus K=0 (uid-paired):")
for r,d in runs.items():
    for name,m in [("Overall",ov),("SC",lambda x:x["SC"]),("PF",lambda x:x["PF"])]:
        mdv,p,lo,hi,n=paired(d,K0,m)
        print(f"  {r:14s} {name:8s} delta {mdv:+.3f}  p {p:.3g}  90%CI [{lo:+.3f},{hi:+.3f}]  (n={n})")
    print()
print("domain vs mismatched @K5 (both degrade catastrophically; the Overall contrast is floor-limited and its components differ, so this does not establish domain-irrelevance):", [f"{n} {round(v,3)}" for n,v in zip(["Overall","SC","PF"], [paired(runs['k5_domain'],runs['k5_mismatched'],m)[0] for m in [ov,lambda x:x['SC'],lambda x:x['PF']]])])
# PF and Overall are censored at 0 at this dose, so the Overall contrast above is floor-limited:
# rows with both PF at 0 contribute an exact zero difference by construction. Reported deltas
# against K=0 are lower bounds on the degradation, not point estimates.
_D, _M = runs["k5_domain"], runs["k5_mismatched"]
_K = sorted(set(_D) & set(_M))
for _n, _d in [("k2_domain", runs["k2_domain"]), ("k5_domain", _D), ("k5_mismatched", _M)]:
    print(f"  censoring {_n:15s} PF=0 {sum(1 for r in _d.values() if r['PF']<=0):3d}/{len(_d)}"
          f"  Overall=0 {sum(1 for r in _d.values() if min(r['PF'],r['SC'])<=0):3d}/{len(_d)}")
_both = [k for k in _K if _D[k]["PF"]<=0 and _M[k]["PF"]<=0]
_un   = [k for k in _K if _D[k]["PF"]>0  and _M[k]["PF"]>0]
print(f"  of {len(_K)} paired rows: both PF floored {len(_both)} ({100*len(_both)/len(_K):.0f}%), "
      f"neither floored {len(_un)} ({100*len(_un)/len(_K):.0f}%)")
_du = [ov(_D[k]) - ov(_M[k]) for k in _un]
print(f"  domain - mismatched Overall on the uncensored rows only: {st.mean(_du):+.3f} "
      f"p {ttest_rel([ov(_D[k]) for k in _un],[ov(_M[k]) for k in _un])[1]:.3f} (n={len(_un)})")
print("  k2 is 18% PF-censored against k5's 60%, so a k2_mismatched arm would test this cleanly")
print("anti-render vs domain (does 'do not draw' help?):", [f"{n} {round(v,3)}" for n,v in zip(["Overall","SC","PF"], [paired(runs['k5_antirender'],runs['k5_domain'],m)[0] for m in [ov,lambda x:x['SC'],lambda x:x['PF']]])])
pf=lambda x:x["PF"]
print("H1 PF decomposition: length effect", round(paired(runs['k5_filler'],K0,pf)[0],3), "(K0->filler) | content-leak effect", round(paired(runs['k5_domain'],runs['k5_filler'],pf)[0],3), "(filler->domain)")
print("real block vs equal-token filler (domain vs filler @K5):", [f"{n} {round(v,3)}" for n,v in zip(["Overall","SC","PF"], [paired(runs['k5_domain'],runs['k5_filler'],m)[0] for m in [ov,lambda x:x['SC'],lambda x:x['PF']]])])
print("dose (k2 vs k5 domain, minus K0):", "k2", round(paired(runs['k2_domain'],K0,ov)[0],3), "| k5", round(paired(runs['k5_domain'],K0,ov)[0],3))

each condition minus K=0 (uid-paired):
  k2_domain      Overall  delta -3.667  p 4.7e-104  90%CI [-3.882,-3.452]  (n=488)
  k2_domain      SC       delta -2.391  p 5.33e-39  90%CI [-2.667,-2.116]  (n=488)
  k2_domain      PF       delta -4.572  p 2.72e-166  90%CI [-4.749,-4.395]  (n=488)

  k5_domain      Overall  delta -5.050  p 1.2e-152  90%CI [-5.263,-4.838]  (n=488)
  k5_domain      SC       delta -2.154  p 1.54e-20  90%CI [-2.519,-1.789]  (n=488)
  k5_domain      PF       delta -6.225  p 7.67e-247  90%CI [-6.379,-6.071]  (n=488)

  k5_mismatched  Overall  delta -5.067  p 1.25e-151  90%CI [-5.282,-4.853]  (n=488)
  k5_mismatched  SC       delta -1.064  p 9.81e-06  90%CI [-1.456,-0.671]  (n=488)
  k5_mismatched  PF       delta -6.486  p 1.62e-256  90%CI [-6.638,-6.333]  (n=488)

  k5_filler      Overall  delta -1.884  p 2.12e-45  90%CI [-2.081,-1.687]  (n=488)
  k5_filler      SC       delta -1.262  p 3.96e-18  90%CI [-1.493,-1.032]  (n=488)
  k5_filler      PF       delta -2.455  p

## Leakage (H1) — validation-gated CLIP
First validate that image-text CLIP can read content on THESE images (own vs random instruction). Only if it separates do we trust the leakage number.

In [ ]:
import torch, glob
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
import torch.nn.functional as F
clip = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to("cuda").eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
test = [json.loads(l) for l in open(f"{DRIVE}/test.jsonl", encoding="utf-8")]
ex   = {json.loads(l)["uid"]: json.loads(l) for l in open(f"{DRIVE}/fewshot_exemplars.jsonl", encoding="utf-8")}
import re as _re
def strip_refs(s): return " ".join(p for p in _re.split(r"(?<=\.)\s+", s) if "<img>" not in p).strip()

@torch.no_grad()
def img_emb(path):
    x = proc(images=Image.open(path).convert("RGB"), return_tensors="pt").to("cuda")
    feat = clip.visual_projection(clip.vision_model(pixel_values=x["pixel_values"]).pooler_output)
    return F.normalize(feat, dim=-1)

@torch.no_grad()
def txt_emb(texts):
    x = proc(text=texts, return_tensors="pt", padding=True, truncation=True, max_length=77).to("cuda")
    feat = clip.text_projection(clip.text_model(input_ids=x["input_ids"], attention_mask=x["attention_mask"]).pooler_output)
    return F.normalize(feat, dim=-1)

# order rows to match {i:05d}.png
GEN0=f"{DRIVE}/gen/test/zero_ref"; GEN5=f"{DRIVE}/gen/test/k5_domain"
N=len(test)   # full test set; a 150-row prefix covers only ~30 contiguous targets
own=[]; rnd=[]; leak=[]
import random; random.seed(0); OFF = random.randint(1, len(test)-1)   # fixed offset -> each image vs a different row's instruction
for i in range(len(test)):
    r=test[i]; uid=r["output_image"]+"|"+"|".join(sorted(r["input_images"]))
    p0=f"{GEN0}/{i:05d}.png"
    if not os.path.exists(p0): continue
    ie0=img_emb(p0)
    # validation: own vs random instruction
    te_own=txt_emb([strip_refs(r["instruction"])[:300]]); te_rnd=txt_emb([strip_refs(test[(i+OFF)%len(test)]["instruction"])[:300]])
    own.append(float((ie0@te_own.T)[0,0])); rnd.append(float((ie0@te_rnd.T)[0,0]))
    # leakage: K5 vs K0 image similarity to the exemplar texts
    p5=f"{GEN5}/{i:05d}.png"
    if os.path.exists(p5):
        te_ex=txt_emb([e[:300] for e in ex[uid]["domain"]]).mean(0, keepdim=True)
        te_ex=F.normalize(te_ex, dim=-1)
        ie5=img_emb(p5)
        leak.append(float((ie5@te_ex.T)[0,0]) - float((ie0@te_ex.T)[0,0]))
    if len(own)>=N: break
print(f"CLIP validation (n={len(own)}): own-instruction sim {st.mean(own):.3f} vs random {st.mean(rnd):.3f} "
      f"(separation {st.mean(own)-st.mean(rnd):+.3f})")
print("=> CLIP can read image-text content here" if st.mean(own)-st.mean(rnd)>0.02 else "=> CLIP is blind here; drop the leakage number")
if leak:
    t,p=ttest_rel([0]*len(leak), [-x for x in leak])  # test leak mean != 0
    clip_p = p                                        # consumed by the decision rule below
    print(f"leakage (CLIP K5-image vs K0-image, to exemplar texts): mean {st.mean(leak):+.4f}  p {p:.3g}  (n={len(leak)})")
    print("=> H1 leakage" if st.mean(leak)>0.01 and p<0.05 else "=> no CLIP-detected leakage")

config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP validation (n=488): own-instruction sim 0.266 vs random 0.158 (separation +0.108)
=> CLIP can read image-text content here
leakage (CLIP K5-image vs K0-image, to exemplar texts): mean +0.0515  p 9.27e-117  (n=488)
=> H1 leakage


## Verdict — decision rule
Decided from the numbers. Eyeball the smoke images to confirm H1. CLIP enters the rule only if it passed its validation gate.

In [ ]:
# decision rule -> prints H1 / H2 / H3 / unresolved
# The H1/H2/H3 labels and the +/-0.15 bound are pre-specified; the CLIP gates below are
# operational thresholds set for this probe. Pre-specified H1 is the leakage rate alone, so the
# h1 disjunction here is wider: its PF arm marks conditioning interference, and the
# rendered-as-content label is carried by the leakage arm.
NOISE = 0.15   # equivalence bound used throughout the thesis
dom_ov = paired(runs["k5_domain"], K0, ov)                       # (delta, p, lo, hi, n)
dom_pf = paired(runs["k5_domain"], K0, lambda x: x["PF"])
dvm    = paired(runs["k5_domain"], runs["k5_mismatched"], ov)    # full tuple: the H3 gate needs p
dom_vs_mis = dvm[0]
clip_leak = (st.mean(leak) if ("leak" in dir() and leak) else None)
clip_valid = ("own" in dir() and len(own) > 0 and (st.mean(own) - st.mean(rnd) > 0.02))

# each branch states its own criterion; nothing falls through to a claim the code did not test
h3 = dom_ov[0] > NOISE and dom_ov[1] < 0.05 and dvm[0] > NOISE and dvm[1] < 0.05
_clip_p = clip_p if "clip_p" in dir() else None
h1 = (dom_pf[0] < -NOISE and dom_pf[1] < 0.05) or (clip_valid and clip_leak and clip_leak > 0.01 and _clip_p is not None and _clip_p < 0.05)
dom_sc = paired(runs["k5_domain"], K0, lambda x: x["SC"])
h2 = (dom_ov[2] > -NOISE and dom_ov[3] < NOISE) and (dom_pf[2] > -NOISE and dom_pf[3] < NOISE) \
     and (dom_sc[2] > -NOISE and dom_sc[3] < NOISE)

if h3:
    verdict = "H3 genuine abstraction — domain improves over K=0 beyond the bound and beats mismatched beyond the bound, both at p<0.05"
elif h1:
    verdict = "H1 rendered-as-content — exemplars displace/leak into the image (PF drop and/or validated CLIP leakage); confirm by eyeballing the smoke images"
elif h2:
    verdict = "H2 ignored — domain equivalent to K=0 within +/-NOISE on Overall, PF and SC (90% CI inside the bound)"
else:
    verdict = "UNRESOLVED — no branch criterion met: neither degradation, abstraction nor equivalence to K=0 is established"

# target-cluster sensitivity: 488 rows come from 100 target scenes and few-shot deltas are
# strongly clustered. Row-level paired means stay the effect estimates; clustered p is robustness.
# clSE carries the G/(G-1) finite-cluster correction; p uses a t(G-1) reference. lora_check and
# soft_prompt_check apply the same SE with a normal reference.
def _clust(cur, ref, metric=ov):
    I = sorted(set(cur) & set(ref))
    d = [metric(cur[u]) - metric(ref[u]) for u in I]
    n = len(d); m = st.mean(d)
    g = {}
    for u, x in zip(I, d): g.setdefault(u.split("|")[0], []).append(x)
    G = len(g)
    se = math.sqrt(sum((sum(v) - len(v) * m) ** 2 for v in g.values()) / n**2 * (G / (G - 1)))
    ssb = sum(len(v) * (st.mean(v) - m) ** 2 for v in g.values())
    ssw = sum(sum((x - st.mean(v)) ** 2 for x in v) for v in g.values())
    kbar = n / G; msb = ssb / (G - 1); msw = ssw / (n - G)
    icc = (msb - msw) / (msb + (kbar - 1) * msw)
    p = 2 * (1 - tdist.cdf(abs(m / se), G - 1))
    return m, icc, se, p, G

print("\ntarget-cluster sensitivity (100 target scenes):")
print(f"  {'contrast':26s} {'mean':>8s} {'ICC':>6s} {'clSE':>7s} {'clustered p':>12s}")
for _lab, _a, _b, _mt in [("K2 vs K=0", runs["k2_domain"], K0, ov),
                          ("K5-domain vs K=0", runs["k5_domain"], K0, ov),
                          ("K5-domain vs K=0 PF", runs["k5_domain"], K0, lambda d: d["PF"]),
                          ("K2-domain vs K=0 PF", runs["k2_domain"], K0, lambda d: d["PF"]),
                          ("filler vs K=0 PF", runs["k5_filler"], K0, lambda d: d["PF"]),
                          ("domain vs filler PF", runs["k5_domain"], runs["k5_filler"], lambda d: d["PF"]),
                          ("K5-mismatched vs K=0", runs["k5_mismatched"], K0, ov),
                          ("filler vs K=0", runs["k5_filler"], K0, ov),
                          ("domain vs filler", runs["k5_domain"], runs["k5_filler"], ov),
                          ("dom-mis PF", runs["k5_domain"], runs["k5_mismatched"], lambda d: d["PF"]),
                          ("dom-mis SC", runs["k5_domain"], runs["k5_mismatched"], lambda d: d["SC"]),
                          ("dom-mis Overall", runs["k5_domain"], runs["k5_mismatched"], ov),
                          ("anti-render PF", runs["k5_antirender"], runs["k5_domain"], lambda d: d["PF"])]:
    _m, _i, _s, _p, _G = _clust(_a, _b, _mt)
    print(f"  {_lab:26s} {_m:>+8.3f} {_i:>6.3f} {_s:>7.3f} {_p:>12.4g}")
print("  clustering leaves every score-derived contrast unchanged, including the PF collapse\n"
      "  that fires the H1 branch. The direct CLIP leakage statistic is iid: its row-level values\n"
      "  were not persisted with uid, so it cannot be cluster-re-analysed from the frozen artifacts.\n")

print("domain vs K=0 Overall:", round(dom_ov[0],3), "p", round(dom_ov[1],4))
print("domain vs K=0 PF     :", round(dom_pf[0],3), "p", round(dom_pf[1],4))
print("domain vs mismatched Overall:", round(dom_vs_mis,3), f"p {round(dvm[1],3)} (floor-limited: both arms collapse, so this is not an equivalence result)")
_ar = paired(runs["k5_antirender"], runs["k5_domain"], lambda x: x["PF"])
print(f"anti-render vs domain PF: {_ar[0]:+.3f}  p {_ar[1]:.3f}  90%CI [{_ar[2]:+.3f},{_ar[3]:+.3f}]"
      "  (no detectable improvement from the tested anti-render framing)")
print("CLIP leakage         :", (round(clip_leak,4) if clip_leak is not None else "n/a"), "| CLIP validated:", clip_valid)
print("\nVERDICT:", verdict)


target-cluster sensitivity (100 target scenes):
  contrast                       mean    ICC    clSE  clustered p
  K2 vs K=0                    -3.667  0.320   0.207            0
  K5-domain vs K=0             -5.050  0.380   0.217            0
  K5-domain vs K=0 PF          -6.225  0.305   0.150            0
  K2-domain vs K=0 PF          -4.572  0.312   0.175            0
  filler vs K=0 PF             -2.455  0.496   0.199            0
  domain vs filler PF          -3.770  0.315   0.174            0
  K5-mismatched vs K=0         -5.067  0.363   0.215            0
  filler vs K=0                -1.884  0.406   0.202    3.553e-15
  domain vs filler             -3.166  0.273   0.186            0
  dom-mis PF                   +0.260  0.135   0.084     0.002453
  dom-mis SC                   -1.090  0.003   0.222     3.71e-06
  dom-mis Overall              +0.017  0.093   0.092        0.855
  anti-render PF               +0.053  0.030   0.059       0.3655
  clustering leaves every s